In [2]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display

In [3]:
# Define the patterns and their corresponding return values
#removed district stock out to ensure that we do not have duplicates in reporting
# pattern_mapping = {
#     r'Was there a stock-out at the national level of ': 'National Stock Out',
#     r'Was there a vaccine stock-out at the district lev of ': 'District Stock Out',
#     r'What was the cause of the national stock-out of ': 'Cause',
#     r'What was the duration of stock-out in months of ': 'Duration'
# }

pattern_mapping = {
    r'Was there a stock-out at the national level of ': 'National Stock Out',
    r'What was the cause of the national stock-out of ': 'Cause',
    r'What was the duration of stock-out in months of ': 'Duration'
}

# Define a function to split the DESCRIPTION into two columns
def split_description(description):
    for pattern, category in pattern_mapping.items():
        if re.search(pattern, description):
            parts = re.split(pattern, description, maxsplit=1)
            details = parts[1].strip().rstrip('?') if len(parts) > 1 else ""
            return category, details
    return description, ""

In [4]:
# Load the sheet that is most likely to contain the relevant data
file_path = "Vaccine supply and logistics 2024-13-06 20-30 UTC.xlsx"
data = pd.read_excel(file_path, sheet_name='Sheet1')
data = data.fillna("Blank")

# Display the first few rows of the dataset to understand its structure
data.head()


,ISO_3_CODE,COUNTRYNAME,WHO_REGION,YEAR,INDCODE,DESCRIPTION,INDCATCODE,INDCAT_DESCRIPTION,INDSORT,VALUE
0,ZMB,Zambia,AFRO,2023,SUPPLY_MANAGMENT_DISCARDED_NATIONAL_RATE,What is the closed-vial wastage rate for DTP-c...,VACCINE_SUPPLY,Vaccine supply and logistics,1078,0
1,ZMB,Zambia,AFRO,2023,SUPPLY_NATIONAL_BCG,Was there a stock-out at the national level of...,VACCINE_SUPPLY,Vaccine supply and logistics,618,Yes
2,ZMB,Zambia,AFRO,2023,SUPPLY_CHAIN_PERCENT,What percentage of cold chain equipment used t...,VACCINE_SUPPLY,Vaccine supply and logistics,1083,0
3,ZMB,Zambia,AFRO,2023,SUPPLY_MM_PCV10,What was the duration of stock-out in months o...,VACCINE_SUPPLY,Vaccine supply and logistics,943,0.75
4,ZMB,Zambia,AFRO,2023,SUPPLY_CAUSE_PCV10,What was the cause of the national stock-out o...,VACCINE_SUPPLY,Vaccine supply and logistics,944,Inaccurate forecasts


In [5]:
# Filter the dataset for rows where the DESCRIPTION column contains the word "stock-out"
stock_out_description = data[data['DESCRIPTION'].str.contains('stock-out', case=False, na=False)].copy()
stock_out_description = stock_out_description[~stock_out_description['DESCRIPTION'].str.contains('home-based vaccination records|district level', case=True, na=False)]
#split DESCRIPTION into event and vaccine
stock_out_description[['Event', 'Vaccine']] = stock_out_description['DESCRIPTION'].apply(split_description).apply(pd.Series)
#string convert for sorting 
stock_out_description['VALUE'] = stock_out_description['VALUE'].astype(str)

#drop uncessary columns
cols_to_drop = ['ISO_3_CODE', 'WHO_REGION','INDCODE',
       'DESCRIPTION', 'INDCATCODE', 'INDCAT_DESCRIPTION', 'INDSORT']
stock_out_description.drop(columns=cols_to_drop, inplace=True)

# Filter out the unwanted 'Event' entries
valid_events = ['National Stock Out', 'Duration', 'Cause']
stock_out_description = stock_out_description[stock_out_description['Event'].isin(valid_events)]

stock_out_description


,COUNTRYNAME,YEAR,VALUE,Event,Vaccine
1,Zambia,2023,Yes,National Stock Out,BCG (Baccille Calmette Guérin) vaccine
3,Zambia,2023,0.75,Duration,PCV-10 (Pneumococcal conjugate vaccine 10-vale...
4,Zambia,2023,Inaccurate forecasts,Cause,PCV-10 (Pneumococcal conjugate vaccine 10-vale...
5,Zambia,2023,1.75,Duration,DTwP-Hib-HepB (Whole cell) vaccine
6,Zambia,2023,Inaccurate forecasts,Cause,DTwP-Hib-HepB (Whole cell) vaccine
...,...,...,...,...,...
47808,Chad,2003,NR,National Stock Out,Hepatitis B containing vaccines
47809,Chad,2003,No,National Stock Out,Measles containing vaccines
47810,Chad,2003,No,National Stock Out,Yellow fever vaccine
47811,Chad,2003,No,National Stock Out,TT (Tetanus toxoid) vaccine


In [6]:
# Pivot the data to aggregate rows based on Country, Year, and Vaccine
pivot_data = stock_out_description.pivot_table(
    index=['COUNTRYNAME', 'YEAR', 'Vaccine'],
    columns='Event',
    values='VALUE',
    aggfunc='first'
).reset_index()

pivot_data

Event,COUNTRYNAME,YEAR,Vaccine,Cause,Duration,National Stock Out
0,Afghanistan,2003,Auto-disable syringes,NaN,Blank,No
1,Afghanistan,2003,BCG (Baccille Calmette Guérin) vaccine,NaN,Blank,No
2,Afghanistan,2003,DTP containing vaccines,NaN,Blank,No
3,Afghanistan,2003,Hepatitis B containing vaccines,NaN,Blank,NR
4,Afghanistan,2003,Hib containing vaccines,NaN,Blank,NR
...,...,...,...,...,...,...
13788,Zimbabwe,2022,Pneumococcal conjugate vaccines,NR,NR,No
13789,Zimbabwe,2022,Rotavirus vaccine,Procurement delays,8,Yes
13790,Zimbabwe,2022,TT / DT / Td containing vaccines,NR,NR,No
13791,Zimbabwe,2022,Typhoid fever conjugate vaccine,NR,NR,No


In [7]:
pivot_data = pivot_data[pivot_data['Vaccine'].str.contains('vaccine', case=False, na=False) | pivot_data['Vaccine'].str.contains('Polio', case=False, na=False)]

pivot_data = pivot_data[~pivot_data['Vaccine'].str.contains('BCG|Japan|Yellow|Typhoid|Hepatitis A|Other|Meningitis', case=False, na=False)]

pivot_data['YEAR'] = pd.to_datetime(pivot_data['YEAR'], format='%Y').dt.strftime('%Y')

In [8]:
pivot_data.Vaccine.unique()

array(['DTP containing vaccines', 'Hepatitis B containing vaccines',
       'Hib containing vaccines', 'Measles containing vaccines',
       'Polio (OPV or IPV)', 'TT (Tetanus toxoid) vaccine',
       'Pneumococcal conjugate vaccines', 'Rotavirus vaccine',
       'IPV (Inactivated polio vaccine)', 'OPV (Oral polio vaccine)',
       'Human Papillomavirus Virus (HPV) vaccines',
       'Meningococcal A conjucate vaccine',
       'Haemophilus influenzae type B (hib) monovalent vaccines',
       'Hepatitis B monovalent vaccines',
       'TT / DT / Td containing vaccines',
       'DTaP-HepB (acellular) vaccine',
       'Hib (Haemophilus influenzae type B) vaccine',
       'MR (Measles and rubella) vaccine',
       'PCV-10 (Pneumococcal conjugate vaccine 10-valent) vaccine',
       'PCV-13 (Pneumococcal conjugate vaccine 13-valent) vaccine',
       'RV-5 (Rotavirus 5-valent) vaccine',
       'DTwP-Hib-HepB (Whole cell) vaccine',
       'HPV-4 (Human Papilloma Virus 4-valent) vaccine',
       

In [9]:
# Function to convert duration values to numeric, keeping non-numeric values unchanged
def convert_duration(value):
    try:
        return pd.to_numeric(value)
    except ValueError:
        return value

# Apply the conversion function to the 'Duration' column
pivot_data['Duration'] = pivot_data['Duration'].apply(convert_duration)

# Update 'Cause' and 'Duration' to 'NONE' where 'National Stock Out' is 'No'
pivot_data.loc[pivot_data['National Stock Out'] == 'No', ['Cause', 'Duration']] = 'NONE'

# Update 'Cause' and 'Duration' to 'NONE' where 'National Stock Out' is 'NR'
pivot_data.loc[pivot_data['National Stock Out'] == 'NR', ['Cause', 'Duration']] = 'NONE'

pivot_data['National Stock Out'] = pivot_data['National Stock Out'].apply(lambda x: 'Yes' if x == 'Yes' else 'No')


In [10]:
pivot_data

Event,COUNTRYNAME,YEAR,Vaccine,Cause,Duration,National Stock Out
2,Afghanistan,2003,DTP containing vaccines,NONE,NONE,No
3,Afghanistan,2003,Hepatitis B containing vaccines,NONE,NONE,No
4,Afghanistan,2003,Hib containing vaccines,NONE,NONE,No
5,Afghanistan,2003,Measles containing vaccines,NONE,NONE,No
6,Afghanistan,2003,Polio (OPV or IPV),NONE,NONE,No
...,...,...,...,...,...,...
13785,Zimbabwe,2022,Measles containing vaccines,NONE,NONE,No
13787,Zimbabwe,2022,OPV (Oral polio vaccine),NONE,NONE,No
13788,Zimbabwe,2022,Pneumococcal conjugate vaccines,NONE,NONE,No
13789,Zimbabwe,2022,Rotavirus vaccine,Procurement delays,8,Yes


In [11]:
result = pivot_data[
    ~pivot_data['Cause'].isin([ 'Blank', 'ND', 'NR','NONE'])
].groupby(['Vaccine', 'YEAR', 'Cause']).count()
result
result.to_excel('pivoted_vax_results.xlsx', index=True)

In [12]:
cleaned_data = result.reset_index()
# cleaned_data['YEAR'] = cleaned_data['YEAR'].astype(int)

# Map unique causes to distinct colors
unique_causes = cleaned_data['Cause'].unique()
unique_causes

array(['Funding delays', 'Inaccurate forecasts', 'Orders not met in full',
       'Procurement delays', 'Cold chain issues', 'Distribution issues',
       'Other/Not identified/Not known', 'Shortage',
       'Stock management issues', 'Global vaccine shortage',
       'Supply delays', 'Quality issue on the vaccine'], dtype=object)

In [15]:
cleaned_data = result.reset_index()
# cleaned_data['YEAR'] = cleaned_data['YEAR'].astype(int)

# Map unique causes to distinct colors
unique_causes = cleaned_data['Cause'].unique()
color_map = {cause: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, cause in enumerate(unique_causes)}

# Define the years for the slider
years = cleaned_data['YEAR'].unique()
vaccines = cleaned_data['Vaccine'].unique()

# Create a trace for each year and color bars according to 'Cause'
traces = []
for year in years:
    filtered_data = cleaned_data[cleaned_data['YEAR'] == year]
    for cause in unique_causes:
        cause_data = filtered_data[filtered_data['Cause'] == cause]
        counts = cause_data['Vaccine'].value_counts().reindex(vaccines, fill_value=0)
        traces.append(go.Bar(
            x=vaccines,
            y=counts,
            marker_color=color_map[cause],
            name=f'{cause} ({year})',
            visible=False
        ))

# Make the first year's data visible
for trace in traces:
    if trace.name.endswith(f'({years[0]})'):
        trace.visible = True

# Create the slider
steps = []
for i, year in enumerate(years):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(traces)},
              {"title": f"Vaccine Distribution by Country for Year {year}"}],
        label=str(year)
    )
    for j in range(len(traces)):
        if traces[j].name.endswith(f'({year})'):
            step["args"][0]["visible"][j] = True  # Toggle j-th trace to be visible
    steps.append(step)

sliders = [dict(
    active=0,
    pad={"t": 50},
    steps=steps
)]

# Create the figure
fig = go.Figure(data=traces)
fig.update_layout(
    sliders=sliders,
    title="Vaccine Distribution by Country",
    xaxis_title="Vaccines",
    yaxis_title="Country Frequency",
    barmode='stack',
    legend=dict(x=1, y=1, title="Cause")
)

fig.show()




In [12]:
import pandas as pd
import plotly.express as px

# Filter and aggregate data
df_filtered = pivot_data[pivot_data['National Stock Out'].isin(['Yes', 'No'])]
df_aggregated = df_filtered.groupby(['COUNTRYNAME', 'National Stock Out', 'YEAR']).size().reset_index(name='Vaccine Count')

# Create the choropleth map with a slider for year
fig = px.choropleth(df_aggregated, locations="COUNTRYNAME",
                    locationmode='country names',
                    color="Vaccine Count",
                    hover_name="COUNTRYNAME",
                    animation_frame="YEAR",
                    facet_col="National Stock Out",
                    color_continuous_scale=px.colors.sequential.Plasma,
                    labels={'Vaccine Count':'Vaccine Count'})

fig.update_geos(showcoastlines=True, coastlinecolor="Black", showland=True, landcolor="white")
fig.update_layout(title_text="Global Vaccine Data - National Stock Out Status",
                  title_x=0.5, geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'))
fig.show()

# Save the figure as an interactive HTML file
fig.write_html('vaccine_stock_out_map.html')

In [33]:
# Filter and aggregate data
df_filtered = pivot_data[pivot_data['National Stock Out'].isin(['Yes', 'No'])]

# Aggregate the data by country, year, and stock out status
df_aggregated = df_filtered.groupby(['COUNTRYNAME', 'YEAR', 'National Stock Out']).size().reset_index(name='Vaccine Count')

# Determine the maximum vaccine count across all years
max_count = df_aggregated['Vaccine Count'].max()

# Calculate the total count of events globally for each year
total_events_per_year = df_aggregated.groupby(['YEAR', 'National Stock Out'])['Vaccine Count'].sum().reset_index(name='Total Events')

# Create the initial choropleth map
fig = px.choropleth(df_aggregated, locations="COUNTRYNAME",
                    locationmode='country names',
                    color="Vaccine Count",
                    animation_frame="YEAR",
                    facet_col="National Stock Out",
                    category_orders={"National Stock Out": ["No", "Yes"]},  # Swapping the order
                    color_continuous_scale=px.colors.sequential.Blues,
                    range_color=(0, max_count),
                    labels={'Vaccine Count':'Vaccine Count'})

fig.update_geos(showcoastlines=True, coastlinecolor="Black", showland=True, landcolor="white")
fig.update_layout(title_text="Global Vaccine Data - National Stock Out Status",
                  title_x=0.5, geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'))

# Initialize the list of frames
frames = []

for year in sorted(total_events_per_year['YEAR'].unique()):
    year_data = df_aggregated[df_aggregated['YEAR'] == year]
    total_yes_events = total_events_per_year[(total_events_per_year['YEAR'] == year) & (total_events_per_year['National Stock Out'] == 'Yes')]['Total Events'].values[0]
    total_no_events = total_events_per_year[(total_events_per_year['YEAR'] == year) & (total_events_per_year['National Stock Out'] == 'No')]['Total Events'].values[0]
    
    frame = go.Frame(
        data=[
            go.Choropleth(
                locations=year_data[year_data['National Stock Out'] == 'No']['COUNTRYNAME'],
                z=year_data[year_data['National Stock Out'] == 'No']['Vaccine Count'],
                locationmode='country names',
                colorscale='Blues',  # Change color scale here
                zmin=0,
                zmax=max_count,
                name="Yes"
            ),
            go.Choropleth(
                locations=year_data[year_data['National Stock Out'] == 'Yes']['COUNTRYNAME'],
                z=year_data[year_data['National Stock Out'] == 'Yes']['Vaccine Count'],
                locationmode='country names',
                colorscale='Blues',  # Change color scale here
                zmin=0,
                zmax=max_count,
                name="No"
            )
        ],
        name=str(year),
        layout=go.Layout(
            annotations=[
                dict(
                    text=f"Year: {year} - Total Yes Events: {total_yes_events}",
                    xref="paper", yref="paper",
                    x=0.75, y=0.95,  # Dynamic position just above the plot
                    showarrow=False,
                    font=dict(size=14),
                    xanchor='center',
                    yanchor='bottom',
                    align='center'
                ),
                dict(
                    text=f"Year: {year} - Total No Events: {total_no_events}",
                    xref="paper", yref="paper",
                    x=0.25, y=0.95,  # Dynamic position just above the plot
                    showarrow=False,
                    font=dict(size=14),
                    xanchor='center',
                    yanchor='bottom',
                    align='center'
                )
            ]
        )
    )
    frames.append(frame)

fig.frames = frames

# Update layout with slider and play button
fig.update_layout(
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}],
                "label": "Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                "label": "Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 20},
            "prefix": "Year:",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 300, "easing": "cubic-in-out"},
        "pad": {"b": 10, "t": 50},
        "len": 0.9,
        "x": 0.1,
        "y": 0,
        "steps": [{
            "args": [[year], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}],
            "label": year,
            "method": "animate"
        } for year in sorted(total_events_per_year['YEAR'].unique())]
    }]
)

fig.show()
# Save the figure as an interactive HTML file
fig.write_html('vaccine_stock_out_map_with_totals.html')


In [51]:
# Filter and aggregate data
df_filtered = pivot_data[pivot_data['National Stock Out'].isin(['Yes', 'No'])]
df_filtered = df_filtered.copy()

# Define the "No" events
no_events = ['NONE', 'nan', 'Blank', 'ND', 'NR']

# Replace nan with a string for consistent handling
df_filtered['National Stock Out'] = df_filtered['National Stock Out'].fillna('nan')

# Filter out the "No" events to focus on "Yes" events
# df_yes_events = df_filtered[~df_filtered['National Stock Out'].isin(no_events)]

df_yes_events = df_yes_events[~df_yes_events['Cause'].isin(['NONE', 'nan', 'Blank', 'ND', 'NR'])]

# Aggregate the data by country, year, and cause
df_aggregated_yes = df_yes_events.groupby(['COUNTRYNAME', 'YEAR', 'Cause']).size().reset_index(name='Frequency')

# Find the cause with the highest frequency for each country per year
df_max_event_yes = df_aggregated_yes.loc[df_aggregated_yes.groupby(['COUNTRYNAME', 'YEAR'])['Frequency'].idxmax()]

# Create a color mapping for the causes
unique_yes_causes = df_max_event_yes['Cause'].unique()
cause_colors = {cause: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)] for i, cause in enumerate(unique_yes_causes)}

# Add year and cause to the data for hover information
df_max_event_yes['hover_text'] = df_max_event_yes['COUNTRYNAME'] + '<br>Year: ' + df_max_event_yes['YEAR'].astype(str) + '<br>Cause: ' + df_max_event_yes['Cause'] + '<br>Frequency: ' + df_max_event_yes['Frequency'].astype(str)

# Create the choropleth map with updated hover information
fig = px.choropleth(df_max_event_yes, locations="COUNTRYNAME",
                    locationmode='country names',
                    color="Cause",
                    hover_name="hover_text",
                    animation_frame="YEAR",
                    color_discrete_map=cause_colors,
                    labels={'Cause':'Event Cause'})

fig.update_geos(showcoastlines=True, coastlinecolor="Black", showland=True, landcolor="white")
fig.update_layout(title_text="Global Vaccine Data - National Stock Out Status by Cause Frequency",
                  title_x=0.5, geo=dict(showframe=False, showcoastlines=True, projection_type='equirectangular'))



fig.show()

# Save the figure as an interactive HTML file
fig.write_html('vaccine_event_frequency_map.html')



## Random Number Generation -  NOT USED CURRENTLY

In [113]:
import random

events_list = [
    'None',
    'Supply Chain Issues',
    'Funding delays',
    'Vaccine shortage',
    'Inaccurate forecasts',
    'Other/Not identified/Not known',
    'Procurement delays',
    'Pandemic'
]

probabilities = [77.51, 0.54, 3.79, 1.59, 0.21, 10.44, 3.37, 2.53]


events_selected = []
i = 0
while i < 10:
    event = random.choices(events_list, probabilities_normalized)[0]
    events_selected.append(event)
    if event == 'Pandemic' and i < 9:
        events_selected.append('Pandemic')
        i += 1
    i += 1

events_selected = events_selected[:10]
events_selected





['None',
 'None',
 'None',
 'Other/Not identified/Not known',
 'Pandemic',
 'Pandemic',
 'None',
 'Other/Not identified/Not known',
 'Funding delays',
 'None']

In [137]:
import random
import pandas as pd

def generate_scenario(events_list, probabilities, affect_list):
    probabilities_normalized = [p / 100 for p in probabilities]

    events_selected = []
    affects_selected = []
    i = 0

    while i < 10:
        event = random.choices(events_list, probabilities_normalized)[0]
        events_selected.append(event)
        
        affect = affect_list[events_list.index(event)]
        affects_selected.append(affect)
        
        if event == 'Pandemic' and i < 9:
            events_selected.append('Pandemic')
            affects_selected.append(affect / 2)
            i += 2
        else:
            i += 1

    events_selected = events_selected[:10]
    affects_selected = affects_selected[:10]

    df = pd.DataFrame({
        'Year': list(range(1, 11)),
        'Event': events_selected,
        'Affect Percent': affects_selected
    })

    return df

# Example usage:
events_list = [
    'None',
    'Supply Chain Issues',
    'Funding delays',
    'Vaccine shortage',
    'Inaccurate forecasts',
    'Other/Not identified/Not known',
    'Procurement delays',
    'Pandemic'
]

probabilities = [84.18, 0.58, 4.12, 1.73, 0.23, 2.75, 3.66, 2.75]
affect_list = [0.00, 10.00, 15.00, 12.00, 8.00, 7.00, 9.00, 40.00]

df = generate_scenario(events_list, probabilities, affect_list)
df



,Year,Event,Affect Percent
0,1,Funding delays,15.0
1,2,None,0.0
2,3,None,0.0
3,4,Pandemic,40.0
4,5,Pandemic,20.0
5,6,None,0.0
6,7,None,0.0
7,8,Inaccurate forecasts,8.0
8,9,Funding delays,15.0
9,10,None,0.0


In [138]:
results = [generate_scenario(events_list, probabilities, affect_list) for _ in range(10)]